In [ ]:
import sys

if "google.colab" in sys.modules:
    !git clone -q https://github.com/jratchfo/learn_ocean_slab.git /content/learn_ocean_slab 2>/dev/null || git -C /content/learn_ocean_slab pull -q
    %pip install -q omegaconf
    sys.path.insert(0, "/content/learn_ocean_slab/src")

# Pollard-Millard slab under a gusty wind: the inertial band

A more realistic wind for the Pollard-Millard slab: a gusty background wind, with one strong
storm near the start. The storm kicks off inertial oscillations, and the gusts keep adding small
kicks of their own.

From the slab current $s(t) = u + iv$ we then keep only the inertial band with a Butterworth
band-pass filter, the same processing one would apply to mixed-layer velocities from S-MODE
(drifters, ADCP or moorings). The run is at 37$^\circ$N, the latitude of the S-MODE site off
California.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, sosfiltfilt

from learn_ocean_slab.config import CONFIG_DIR, SECONDS_PER_DAY, load_config
from learn_ocean_slab.forcing import wind_gusts
from learn_ocean_slab.pm_slab import integrate, step_exact, step_rk4

## Wind forcing

- **Background:** a steady 0.05 N/m$^2$ plus red-noise gusts (standard deviation 0.03 N/m$^2$ per
  component, lasting about 3 hours), from `wind_gusts`.
- **Storm:** a Gaussian burst peaking at 0.4 N/m$^2$ on day 1.5, with a 3-hour standard deviation (about 7 hours above half strength).
  Short compared with the 20-hour inertial period, so it kicks hard.

Both blow toward the south-southeast, like the upwelling-favorable winds off California.
The storm sits a day and a half in, not at $t = 0$, so the filter has quiet water before it.

In [ ]:
config = load_config(CONFIG_DIR / "midlat_45N.yaml", ["lat=37", "duration_days=20", "u_initial=0", "v_initial=0"])
t = np.arange(0, config.duration, config.dt)  # [s]
days = t / SECONDS_PER_DAY

direction = np.exp(-1j * np.radians(60))  # 60 degrees clockwise from east: toward the south-southeast
storm = 0.4 * np.exp(-0.5 * ((t - 1.5 * SECONDS_PER_DAY) / (3 * 3600)) ** 2)  # [N/m^2]
background = 0.05 + wind_gusts(t, gust_std=0.03, gust_time=3 * 3600, seed=config.seed)  # [N/m^2]
tau = (storm + background) * direction  # wind stress tau_x + i tau_y [N/m^2]

fig, ax = plt.subplots(figsize=(12, 3), layout="constrained")
ax.plot(days, tau.real, lw=1, label=r"$\tau_x$")
ax.plot(days, tau.imag, lw=1, label=r"$\tau_y$")
ax.axhline(0, color="0.6", lw=0.8, zorder=0)
ax.set(xlabel="time [days]", ylabel="wind stress [N/m$^2$]", title="Gusty wind with a storm on day 1.5")
ax.legend(loc="upper right")

## Slab response

The wind series goes straight into `integrate`, the same exact per-interval stepper `run` uses,
with $H$ = 50 m and a 5-day damping time from the config.

In [ ]:
s_0 = config.u_initial + 1j * config.v_initial
depths = [25.0, 75.0]  # mixed layer depths [m]
colors = {25.0: "C0", 75.0: "C3"}
runs = {}  # H -> slab current s [m/s]

fig, ax = plt.subplots(figsize=(6, 6), layout="constrained")
for H in depths:
    s, position, wind_input, damping_loss, shear_exchange = integrate(
        t, tau, config.rho, H, config.f, config.r, config.zeta, s_0
    )
    runs[H] = s
    ax.plot(position.real / 1e3, position.imag / 1e3, color=colors[H], lw=1, label=f"$H$ = {H:g} m")
ax.set(xlabel="east [km]", ylabel="north [km]", title="Parcel paths", aspect="equal")
ax.legend()

## Does the stepper matter?

`integrate`'s default, `step_exact`, holds the wind at each interval's *average* and solves the
Pollard-Millard equation exactly for that constant forcing. That's exact for the forcing it's given,
but the forcing itself jumps from one interval's average to the next's, so $du/dt$ has a jump at
every grid point, however small `dt` is.

`step_rk4` interpolates the wind linearly between grid points instead, so the forcing - and $du/dt$ -
stays continuous, and integrates with 4th-order Runge-Kutta rather than an exact per-interval solution.

Comparing both at $H$ = `config.H`, zoomed on the storm (where the wind changes fastest, so any
discontinuity is easiest to see).

In [ ]:


s_exact, *_ = integrate(t, tau, config.rho, config.H, config.f, config.r, config.zeta, s_0, step=step_exact, forcing="average")
s_rk4, *_ = integrate(t, tau, config.rho, config.H, config.f, config.r, config.zeta, s_0, step=step_rk4, forcing="linear")

# What each stepper actually assumes the wind is doing between grid points: step_exact holds it at
# each interval's average (a staircase); step_rk4 interpolates it linearly (the line through the
# samples). This staircase, not either integrator's accuracy, is where the du/dt discontinuity comes from.
tau_mid = (tau[:-1] + tau[1:]) / 2
days_mid = 0.5 * (days[:-1] + days[1:])

window = (days > 1.3) & (days < 1.7)  # zoom on the storm, where tau changes fastest
window_mid = (days_mid > 1.3) & (days_mid < 1.7)

fig, (ax_s, ax_tau) = plt.subplots(2, 1, figsize=(10, 7), layout="constrained", sharex=True)
ax_s.plot(days[window], s_exact.real[window], color="C0", label="step_exact (average forcing)")
ax_s.plot(days[window], s_rk4.real[window], color="C3", ls="--", label="step_rk4 (linear forcing)")
ax_s.set(ylabel="eastward velocity [m/s]", title="Current, zoomed on the storm")
ax_s.legend(loc="upper right")

ax_tau.step(
    days_mid[window_mid], tau_mid.real[window_mid], where="mid", color="C0",
    label=r"step_exact's $\tau_x$ (held per interval)",
)
ax_tau.plot(
    days[window], tau.real[window], color="C3", ls="--", marker=".", ms=4,
    label=r"step_rk4's $\tau_x$ (linear between samples)",
)
ax_tau.set(xlabel="time [days]", ylabel=r"$\tau_x$ [N/m$^2$]", title="The forcing each stepper actually sees")
ax_tau.legend(loc="upper right")

print(f"max |s_exact - s_rk4| over the run: {np.max(np.abs(s_exact - s_rk4)):.2e} m/s")
print(f"typical current speed |s|: {np.mean(np.abs(s_exact)):.3f} m/s")

## Butterworth band-pass around $f$

At 37$^\circ$N the inertial period is $2\pi/f \approx$ 19.9 hours. The filter keeps frequencies
between $0.8f$ and $1.2f$. It runs on $u$ and $v$ separately; the filter is linear, so this is the
same as filtering $s = u + iv$.

A narrow band means a slow filter: with a bandwidth of $0.4f$ it takes about $2\pi/(0.4f)$, two and a
half inertial periods (2 days here), to respond. So the band-passed current builds up over that time
after the storm rather than jumping, and the first and last 2 days of the record (shaded) are
unreliable. The same is true at the ends of an observed record and at gaps in it.

`sosfiltfilt` runs the filter forward and then backward, so the result has no phase shift (peaks
stay at the right times) and its response is the Butterworth response squared.

On the spectrum, a frequency $\omega$ means $s \propto e^{i\omega t}$. Inertial oscillations turn
clockwise, $s \propto e^{-ift}$, so their peak sits at $\omega = -f$.

In [ ]:
def inertial_bandpass(s, dt, f, low=0.8, high=1.2, order=4):
    # Butterworth band-pass from low*|f| to high*|f|, run forward and backward (no phase shift)
    band = np.array([low, high]) * abs(f) / (2 * np.pi)  # [Hz]
    sos = butter(order, band, btype="bandpass", fs=1 / dt, output="sos")
    return sosfiltfilt(sos, s.real) + 1j * sosfiltfilt(sos, s.imag)


# Rotary spectrum: negative frequencies turn clockwise
frequency = np.fft.fftshift(np.fft.fftfreq(len(t), config.dt)) * 2 * np.pi / config.f  # [omega / f]

fig, (ax_spec, ax_u, ax_speed) = plt.subplots(3, 1, figsize=(12, 10), layout="constrained")
for H, s in runs.items():
    s_inertial = inertial_bandpass(s, config.dt, config.f)
    power = np.abs(np.fft.fftshift(np.fft.fft(s - s.mean()))) ** 2
    power_inertial = np.abs(np.fft.fftshift(np.fft.fft(s_inertial))) ** 2
    color = colors[H]
    ax_spec.semilogy(frequency, power, color=color, lw=0.8, alpha=0.4)
    ax_spec.semilogy(frequency, power_inertial, color=color, lw=1.2, label=f"$H$ = {H:g} m, band-passed")
    ax_u.plot(days, s.real, color=color, lw=0.8, alpha=0.4)
    ax_u.plot(days, s_inertial.real, color=color, lw=1.2, label=f"$H$ = {H:g} m")
    ax_speed.plot(days, np.abs(s), color=color, lw=0.8, alpha=0.4)
    ax_speed.plot(days, np.abs(s_inertial), color=color, lw=1.5, label=f"$H$ = {H:g} m")

for edge in (-1.2, -0.8, 0.8, 1.2):
    ax_spec.axvline(edge, color="k", ls=":", lw=0.8)
ax_spec.set(
    xlim=(-3, 3),
    xlabel=r"frequency $\omega / f$ (negative = clockwise)",
    ylabel="power [arbitrary]",
    title=r"Rotary spectrum (faint: unfiltered); dotted: filter band $0.8 \leq |\omega|/f \leq 1.2$",
)
ax_spec.set_ylim(bottom=ax_spec.get_ylim()[1] * 1e-9)
ax_spec.legend(loc="upper right")

ax_u.axhline(0, color="0.6", lw=0.8, zorder=0)
ax_u.set(ylabel="eastward velocity [m/s]", title="Eastward current (faint: unfiltered; solid: inertial band)")
ax_u.legend(loc="upper right")

ax_speed.set(
    xlabel="time [days]",
    ylabel="speed [m/s]",
    title="Speed (faint: unfiltered; solid: inertial band)",
)
ax_speed.legend(loc="upper right")
edge = 2.5 * config.inertial_period / SECONDS_PER_DAY  # filter response time [days]
for ax in (ax_u, ax_speed):
    ax.set_xlim(days[0], days[-1])
    ax.axvspan(days[0], days[0] + edge, color="0.9", zorder=0)
    ax.axvspan(days[-1] - edge, days[-1], color="0.9", zorder=0)